In [3]:
# %load_ext autoreload
# %autoreload 2

import os
import datasets
import logging
import json
import threading
import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from dotenv import load_dotenv
# from langchain.docstore.document import Document
from langchain_core.documents import Document
# from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_core.vectorstores import VectorStore
from smolagents import OpenAIServerModel, CodeAgent, Tool
from smolagents.monitoring import LogLevel
from transformers import AutoTokenizer
from tqdm import tqdm
from typing import List, Optional
from utils.blablador_helper import BlabladorChatModel
import yaml


e:\MyDoc\Deep_Thinkvitality\MLcode\RAG_paper\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [4]:
##### Load or create vector database parallelly


def sanitize_filename(dataset_name: str) -> str:
    """Convert dataset name to a valid filename by replacing '/' with '_'"""
    return dataset_name.replace("/", "_")


class DocumentProcessor:
    """Thread-safe document processor to avoid pickling issues"""

    def __init__(self, chunk_size: int = 200, chunk_overlap: int = 20):
        self.text_splitter = None
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self._lock = threading.Lock()

    def _get_text_splitter(self):
        """Initialize text splitter in thread-local manner"""
        if self.text_splitter is None:
            with self._lock:
                if self.text_splitter is None:
                    self.text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
                        AutoTokenizer.from_pretrained("thenlper/gte-small"),
                        chunk_size=self.chunk_size,
                        chunk_overlap=self.chunk_overlap,
                        add_start_index=True,
                        strip_whitespace=True,
                        separators=["\n\n", "\n", ".", " ", ""],
                    )
        return self.text_splitter

    def split_documents_chunk(self,
                              docs_chunk: List[Document]) -> List[Document]:
        """Split a chunk of documents using the text splitter"""
        text_splitter = self._get_text_splitter()
        processed_docs = []
        for doc in docs_chunk:
            new_docs = text_splitter.split_documents([doc])
            processed_docs.extend(new_docs)
        return processed_docs


def parallel_document_splitting(
        source_docs: List[Document],
        max_workers: int = None,
        chunk_size: int = 100,
        text_chunk_size: int = 200,
        text_chunk_overlap: int = 20) -> List[Document]:
    """Split documents in parallel using ThreadPoolExecutor"""
    if max_workers is None:
        max_workers = min(8, len(source_docs) // chunk_size + 1)

    print(f"Splitting documents in parallel using {max_workers} threads...")

    # Split source_docs into chunks for parallel processing
    doc_chunks = [
        source_docs[i:i + chunk_size]
        for i in range(0, len(source_docs), chunk_size)
    ]

    # Create a shared processor instance with configurable chunk size
    processor = DocumentProcessor(text_chunk_size, text_chunk_overlap)

    # Use ThreadPoolExecutor instead of ProcessPoolExecutor
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Process chunks in parallel with progress bar
        results = list(
            tqdm(executor.map(processor.split_documents_chunk, doc_chunks),
                 total=len(doc_chunks),
                 desc="Processing document chunks"))

    # Flatten results
    all_processed_docs = []
    for chunk_result in results:
        all_processed_docs.extend(chunk_result)

    return all_processed_docs


def remove_duplicates(docs: List[Document]) -> List[Document]:
    """Remove duplicate documents based on page_content"""
    print("Removing duplicate documents...")
    unique_texts = set()
    unique_docs = []

    for doc in tqdm(docs, desc="Checking for duplicates"):
        if doc.page_content not in unique_texts:
            unique_texts.add(doc.page_content)
            unique_docs.append(doc)

    print(f"Removed {len(docs) - len(unique_docs)} duplicate documents")
    return unique_docs


def batch_embed_documents(docs: List[Document],
                          embedding_model,
                          batch_size: int = 100) -> FAISS:
    """Create FAISS vectorstore with batch processing for embeddings"""
    print(f"Embedding documents in batches of {batch_size}...")

    if not docs:
        raise ValueError("No documents to embed")

    # Create initial vectorstore with first batch
    first_batch = docs[:batch_size]
    print(f"Creating initial vectorstore with {len(first_batch)} documents...")

    vectordb = FAISS.from_documents(
        documents=first_batch,
        embedding=embedding_model,
        distance_strategy=DistanceStrategy.COSINE,
    )

    # Process remaining documents in batches
    remaining_docs = docs[batch_size:]
    if remaining_docs:
        total_batches = (len(remaining_docs) + batch_size - 1) // batch_size

        for i in tqdm(range(0, len(remaining_docs), batch_size),
                      desc="Adding document batches",
                      total=total_batches):
            batch = remaining_docs[i:i + batch_size]

            # BUG FIX: Directly inject the embed batch into the existing FAISS index 
            # instead of creating a new DB & merging indices.
            vectordb.add_documents(batch)

    return vectordb



def sequential_document_splitting(
        source_docs: List[Document],
        text_chunk_size: int = 200,  # Add this parameter
        text_chunk_overlap: int = 20) -> List[Document]:
    """Fallback sequential document splitting if parallel processing fails"""
    print("Using sequential document splitting...")

    text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
        AutoTokenizer.from_pretrained("thenlper/gte-small"),
        chunk_size=text_chunk_size,
        chunk_overlap=text_chunk_overlap,
        add_start_index=True,
        strip_whitespace=True,
        separators=["\n\n", "\n", ".", " ", ""],
    )

    docs_processed = []
    for doc in tqdm(source_docs, desc="Splitting documents"):
        new_docs = text_splitter.split_documents([doc])
        docs_processed.extend(new_docs)

    return docs_processed


def load_or_create_vectordb(dataset_name: str,
                            batch_size: int = 100,
                            max_workers: int = None,
                            doc_chunk_size: int = 100,
                            text_chunk_size: int = 200,
                            text_chunk_overlap: int = 20,
                            force_rebuild: bool = False,
                            use_parallel: bool = True) -> FAISS:
    """Load existing vectordb or create new one with optimized processing"""

    # Create sanitized filename
    safe_filename = sanitize_filename(dataset_name)
    vectordb_path = os.path.join(
        "vectordb",
        f"{safe_filename}_{text_chunk_size}")  # Include chunk size in filename

    # Check if vectordb already exists and not forcing rebuild
    if os.path.exists(vectordb_path) and not force_rebuild:
        print(f"Found existing vector database at {vectordb_path}")
        print("Loading existing vector database...")

        try:
            # Load the embedding model (needed for loading)
            embedding_model = HuggingFaceEmbeddings(
                model_name="thenlper/gte-small")

            # Load the existing vectordb
            vectordb = FAISS.load_local(vectordb_path,
                                        embedding_model,
                                        allow_dangerous_deserialization=True)
            print("Vector database loaded successfully!")
            return vectordb

        except Exception as e:
            print(f"Error loading existing vectordb: {e}")
            print("Creating new vector database...")

    else:
        if force_rebuild:
            print("Force rebuild requested. Creating new vector database...")
        else:
            print(f"No existing vector database found at {vectordb_path}")
            print("Creating new vector database...")

    # Load and process the dataset
    print("Loading dataset...")
    knowledge_base = datasets.load_dataset(dataset_name, split="train")
    source_docs = [
        Document(page_content=doc["text"],
                 metadata={"source": doc["source"].split("/")[1]})
        for doc in tqdm(knowledge_base, desc="Processing dataset")
    ]

    print(f"Loaded {len(source_docs)} documents from dataset")

    # Split documents (with fallback to sequential processing)
    try:
        if use_parallel and len(source_docs) > 50:
            docs_processed = parallel_document_splitting(
                source_docs, max_workers, doc_chunk_size, text_chunk_size,
                text_chunk_overlap)
        else:
            docs_processed = sequential_document_splitting(
                source_docs, text_chunk_size, text_chunk_overlap)

    except Exception as e:
        print(f"Parallel processing failed: {e}")
        print("Falling back to sequential processing...")
        docs_processed = sequential_document_splitting(
            source_docs, text_chunk_size,
            text_chunk_overlap)  # Pass new parameters

    print(f"Split into {len(docs_processed)} document chunks")

    # Remove duplicates
    docs_processed = remove_duplicates(docs_processed)

    print(f"Final document count after deduplication: {len(docs_processed)}")

    # Create embedding model
    print("Loading embedding model...")
    embedding_model = HuggingFaceEmbeddings(model_name="thenlper/gte-small")

    # Create vectordb with batch processing
    vectordb = batch_embed_documents(docs_processed, embedding_model,
                                     batch_size)

    # Save the vectordb
    print(f"Saving vector database to {vectordb_path}...")
    try:
        vectordb.save_local(vectordb_path)
        print("Vector database saved successfully!")
    except Exception as e:
        print(f"Error saving vectordb: {e}")
        print("Continuing without saving...")

    return vectordb


# Usage example with configurable parameters
dataset_name = "m-ric/huggingface_doc"

# Usage example with configurable chunk sizes
config = {
    "batch_size": 50,
    "max_workers": 4,
    "doc_chunk_size": 100,  # Parallel processing batch size
    "text_chunk_size": 200,  # Document content chunk size (tokens) 
    "text_chunk_overlap": 40,  # Overlap between chunks
    "force_rebuild": False,
    "use_parallel": True
}

vectordb = load_or_create_vectordb(dataset_name, **config)


Found existing vector database at vectordb\m-ric_huggingface_doc_200
Loading existing vector database...


C:\Users\Philippe\AppData\Local\Temp\ipykernel_26968\4134665990.py:180: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: thenlper/gte-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector database loaded successfully!


In [5]:
class RetrieverTool(Tool):
    name = "retriever"
    # description = """
    # Retrieves relevant documents from the knowledge base using semantic similarity search.
    # Choose k based on query complexity:
    # - k=5-7: For standard questions needing some context  
    # - k=8-10: For complex topics requiring multiple perspectives
    # - k=10-13: For comprehensive research or broad exploration
    # """

    # inputs = {
    #     "query": {
    #         "type":
    #         "string",
    #         "description":
    #         "The search query. Use descriptive keywords and phrases rather than questions. Example: 'machine learning algorithms' instead of 'what are ML algorithms?'",
    #     },
    #     "k": {
    #         "type": "integer",
    #         "description":
    #         "Number of documents to retrieve (7-20). Start with 7, increase if no relevant information is found. Default: 7 if not specified.",
    #         "nullable": True
    #     }
    # }

    description = """
    Retrieves relevant documents from the knowledge base using semantic similarity search.
    Choose k based on query complexity.
    """

    inputs = {
        "query": {
            "type":
            "string",
            "description":
            "The search query. Use descriptive keywords and phrases rather than questions. Example: 'machine learning algorithms' instead of 'what are ML algorithms?'",
        },
        "k": {
            "type": "integer",
            "description":
            "Number of documents to retrieve (3-8). Start with 3, increase if no relevant information is found. Default: 3 if not specified.",
            "nullable": True
        }
    }
    output_type = "string"

    def __init__(self,
                 vectordb: VectorStore,
                 k: int = 7,
                 score_threshold: Optional[float] = None,
                 max_content_length: int = 10000,
                 **kwargs):
        super().__init__(**kwargs)
        self.vectordb = vectordb
        self.k = k
        self.score_threshold = score_threshold
        self.max_content_length = max_content_length
        self.logger = logging.getLogger(self.__class__.__name__)

    def forward(self, query: str, k: Optional[int] = None) -> str:
        # Input validation
        if not query or not isinstance(query, str) or not query.strip():
            return "Error: Query must be a non-empty string"

        # Parameter validation
        retrieval_k = k if k is not None else self.k
        if retrieval_k is not None and (not isinstance(retrieval_k, int) or
                                        retrieval_k < 3 or retrieval_k > 20):
            retrieval_k = self.k

        query = query.strip()

        try:
            # Try to get documents with scores if available
            if hasattr(self.vectordb, 'similarity_search_with_score'
                       ) and self.score_threshold is not None:
                docs_with_scores = self.vectordb.similarity_search_with_score(
                    query, k=retrieval_k)
                # Filter by score threshold
                filtered_docs = [
                    doc for doc, score in docs_with_scores
                    if score >= self.score_threshold
                ]
                docs = filtered_docs[:retrieval_k] if filtered_docs else []

                if not docs and docs_with_scores:
                    # If no docs meet threshold, take the best one anyway
                    docs = [docs_with_scores[0][0]]

            else:
                docs = self.vectordb.similarity_search(query, k=retrieval_k)

        except Exception as e:
            self.logger.error(f"Error during similarity search: {str(e)}")
            return f"Error retrieving documents: {str(e)}"

        # Handle no results
        if not docs:
            return f"No relevant documents found for query: '{query}'"

        # Format results
        return self._format_results(query, docs)

    def _format_results(self, query: str, docs) -> str:
        """Format the retrieved documents into a readable string."""
        result_parts = [
            f"Search Query: {query}",
            f"Retrieved {len(docs)} relevant document(s):", ""
        ]

        for i, doc in enumerate(docs, 1):
            # Add document header
            result_parts.append(f"=== Document {i} ===")

            # Add metadata if available
            if hasattr(doc, 'metadata') and doc.metadata:
                metadata_info = []
                if 'source' in doc.metadata:
                    metadata_info.append(f"Source: {doc.metadata['source']}")
                if 'title' in doc.metadata:
                    metadata_info.append(f"Title: {doc.metadata['title']}")
                if 'page' in doc.metadata:
                    metadata_info.append(f"Page: {doc.metadata['page']}")

                if metadata_info:
                    result_parts.append(" | ".join(metadata_info))

            # Add content (with truncation if needed)
            content = doc.page_content
            if len(content) > self.max_content_length:
                content = content[:self.max_content_length].rsplit(
                    ' ', 1)[0] + "... [truncated]"

            result_parts.append(content)
            result_parts.append("")  # Empty line between documents

        return "\n".join(result_parts).strip()

    def get_retrieval_stats(self) -> dict:
        """Get basic stats about the vector store if available."""
        try:
            # This depends on the vector store implementation
            if hasattr(self.vectordb, '__len__'):
                return {"total_documents": len(self.vectordb)}
            return {"status": "Vector store connected"}
        except Exception:
            return {"status": "Unable to retrieve stats"}

In [13]:
# from dev.blabla_test import API_KEY
##### LLM model initialization

load_dotenv()

# Clear any conflicting OpenAI environment variables that might override api_base
# if 'OPENAI_BASE_URL' in os.environ:
#     del os.environ['OPENAI_BASE_URL']
# if 'OPENAI_API_BASE' in os.environ:
#     del os.environ['OPENAI_API_BASE']
    
# model_name = "gemini-2.5-flash-preview-05-20"
# # model_name = "gemini-1.5-flash"
# answer_llm = OpenAIServerModel(
#     model_id=model_name,
#     api_base="https://generativelanguage.googleapis.com/v1beta/openai/",
#     api_key=os.getenv("Gemini_API_KEY"),
#     temperature=0.2)

API_KEY= os.getenv("Blablador_API_KEY")
LLM = BlabladorChatModel(api_key=API_KEY)
model_name = "Ministral" #Qwen3.5 122B, MiniMax-M2.5, NVIDIA-Nemotron
model_fullname = LLM.get_model_fullname(model_name)
print(f"The RAG uses the following model: {model_fullname}\n")

answer_llm = OpenAIServerModel(
    model_id=model_fullname,
    api_base="https://api.helmholtz-blablador.fz-juelich.de/v1",
    api_key=API_KEY,
    max_tokens=16384,
    # flatten_messages_as_text=True,
    temperature=0.2)
retriever_tool = RetrieverTool(vectordb)

agent = CodeAgent(
    tools=[retriever_tool],
    model=answer_llm,
    planning_interval=5,
    max_steps=30,
    verbosity_level=LogLevel.INFO,
)

Model 0 - Ministral-3-14B-Instruct-2512 - The latest Ministral from Dec.2.2025 found
The RAG uses the following model: 0 - Ministral-3-14B-Instruct-2512 - The latest Ministral from Dec.2.2025



In [7]:
eval_dataset = datasets.load_dataset("m-ric/huggingface_doc_qa_eval",
                                     split="train")

In [ ]:
##### Using agentic RAG to answer questions

outputs_agentic_rag = []
prompt_filename = "guide_agent_system_prompt.yaml"
prompt_path = os.path.join("prompts", prompt_filename)
with open(prompt_path, "r", encoding="utf-8") as f:
    prompt_config = yaml.safe_load(f)

for example in tqdm(eval_dataset):
    question = example["question"]
    enhanced_question = prompt_config["prompt"].format(question=question)

    # enhanced_question = f"""Use the 'retriever' tool to find information and answer this question: {question}

    # SEARCH STRATEGY:
    # 1. Start with k=7 using your best keywords from the question
    # 2. If information is incomplete or irrelevant, retry with k=10 (same keywords)
    # 3. If the same documents are retrieved and information is still insufficient, try different keywords:
    # - Use synonym words from the question to find different documents
    # - Reduce keywords to search for broader documents
    # - Use related concepts from previously retrieved documents
    # 4. Repeat steps 1-3 with new keyword combinations until you find the answer

    # SEARCH RULES:
    # - Do NOT include "Hugging Face" in queries (all questions relate to Hugging Face ecosystem)
    # - The answer exists in the database - keep searching with different approaches
    # - Try both specific and broad keyword strategies

    # RESPONSE RULES:
    # - Answer strictly based on retrieved documents, not prior knowledge
    # - If answer is found: provide clear, direct response with relevant details
    # - If answer not found: summarize the most relevant information retrieved

    # Begin your search systematically.
    # """
    
    # enhanced_question = f"""
    # """
    answer = agent.run(enhanced_question)
    print("=======================================================")
    print(f"Question: {question}")
    print(f"Answer: {answer}")
    print(f'True answer: {example["answer"]}')

    results_agentic = {
        "question": question,
        "true_answer": example["answer"],
        "source_doc": example["source_doc"],
        "generated_answer": answer,
    }
    outputs_agentic_rag.append(results_agentic)
    # time.sleep(5)

  0%|          | 0/65 [00:00<?, ?it/s]

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Use the 'retriever' tool to find information and answer this question: What architecture is the                 │
│ `tokenizers-linux-x64-musl` binary designed for?                                                                │
│                                                                                                                 │
│                                                                                                                 │
│   **Search Strategy:**                                                                                          │
│   1. **Initial Query**: Start with k=5 using the most relevant keywords.                                        │
│   2. **Refinement**: If the information is insufficient, retry with an increased k or alternative keywords      │
│ (synonyms, broader terms, or related concepts).                                                                 │
│   3. **Iteration**: Continue searching with new keyword combinations until the answer is located.               │
│                                                                                                                 │
│   **Search Rules:**                                                                                             │
│   - Do NOT include "Hugging Face" in queries (the context is already focused on it).                            │
│   - The information exists in the database; explore multiple search paths if needed.                            │
│                                                                                                                 │
│   **Response Guidelines:**                                                                                      │
│   - Answer strictly based on retrieved documents, not prior knowledge.                                          │
│   - Give ONLY the final answer in as few words as possible. No explanation, no reasoning, no quoting, no        │
│ context.                                                                                                        │
│   - Extract the most precise and complete answer span from the document. Do not truncate or shorten the key     │
│ term.                                                                                                           │
│   - Provide a concise and accurate response if the answer is found.                                             │
│   - If the answer remains elusive, summarize the most relevant context retrieved.                               │
│                                                                                                                 │
│   Proceed systematically with your search.                                                                      │
│                                                                                                                 │
╰─ OpenAIModel - 0 - Ministral-3-14B-Instruct-2512 - The latest Ministral from Dec.2.2025 ────────────────────────╯

In [6]:
##### Set a checkpoint for query limitations (limited API) to use agentic RAG to answer questions


def run_agentic_rag_with_checkpointing(
        eval_dataset,
        agent,
        checkpoint_file="checkpoints/rag_checkpoint.json"):
    """
    Run agentic RAG with checkpointing to allow resuming from interruptions.
    
    Args:
        eval_dataset: Dataset containing questions and answers
        agent: The agent to run queries with
        checkpoint_file: File to store checkpointing information
    
    Returns:
        List of results from agentic RAG
    """
    outputs_agentic_rag = []
    start_idx = 0

    # Load checkpoint if exists
    if os.path.exists(checkpoint_file):
        try:
            with open(checkpoint_file, 'r') as f:
                checkpoint_data = json.load(f)
                outputs_agentic_rag = checkpoint_data.get('results', [])
                start_idx = checkpoint_data.get('next_idx', 0)
                print(
                    f"Resuming from checkpoint at index {start_idx} with {len(outputs_agentic_rag)} results already processed"
                )
        except Exception as e:
            print(f"Error loading checkpoint: {e}")
            print("Starting from beginning")

    # Create progress bar starting from checkpoint
    total_examples = len(eval_dataset)
    pbar = tqdm(total=total_examples, initial=start_idx)

    try:
        for idx in range(start_idx, total_examples):
            example = eval_dataset[idx]
            question = example["question"]

            enhanced_question = f"""Use the 'retriever' tool to find information and answer this question: {question}

            **Search Strategy:**
            1. **Initial Query**: Start with k=5 using the most relevant keywords.
            2. **Refinement**: If the information is insufficient, retry with an increased k or alternative keywords (synonyms, broader terms, or related concepts).
            3. **Iteration**: Continue searching with new keyword combinations until the answer is located.

            **Search Rules:**
            - Do NOT include "Hugging Face" in queries (the context is already focused on it).
            - The information exists in the database; explore multiple search paths if needed.

            **Response Guidelines:**
            - Answer strictly based on retrieved documents, not prior knowledge.
            - Provide a concise and accurate response if the answer is found.
            - If the answer remains elusive, summarize the most relevant context retrieved.

            Proceed systematically with your search."""

            try:
                answer = agent.run(enhanced_question)

                print(
                    "=======================================================")
                print(f"Question: {question}")
                print(f"Answer: {answer}")
                print(f'True answer: {example["answer"]}')

                results_agentic = {
                    "question": question,
                    "true_answer": example["answer"],
                    "source_doc": example["source_doc"],
                    "generated_answer": answer,
                }

                outputs_agentic_rag.append(results_agentic)

                # Update checkpoint after each successful query
                save_checkpoint(checkpoint_file, outputs_agentic_rag, idx + 1)

                pbar.update(1)
                # time.sleep(5)  # Rate limiting

            except Exception as e:
                print(f"Error processing example {idx}: {e}")
                # Save checkpoint before exiting
                save_checkpoint(checkpoint_file, outputs_agentic_rag, idx)
                print(f"Checkpoint saved. Resume from index {idx}")
                raise

    except KeyboardInterrupt:
        print("\nProcess interrupted by user")
        # Save checkpoint before exiting
        save_checkpoint(checkpoint_file, outputs_agentic_rag, idx)
        print(f"Checkpoint saved. Resume from index {idx}")

    finally:
        pbar.close()

    return outputs_agentic_rag


def save_checkpoint(checkpoint_file, results, next_idx):
    """Save checkpoint data to file"""
    checkpoint_data = {
        "results": results,
        "next_idx": next_idx,
        "timestamp": time.time()
    }

    # Use a temporary file to avoid corruption if interrupted during writing
    temp_file = f"{checkpoint_file}.temp"
    with open(temp_file, 'w') as f:
        json.dump(checkpoint_data, f)

    # Safely replace the old checkpoint file
    if os.path.exists(checkpoint_file):
        os.replace(temp_file, checkpoint_file)
    else:
        os.rename(temp_file, checkpoint_file)


outputs_agentic_rag = run_agentic_rag_with_checkpointing(eval_dataset, agent)

Resuming from checkpoint at index 49 with 49 results already processed


 75%|███████▌  | 49/65 [06:04<?, ?it/s]

Error processing example 49: <!DOCTYPE HTML PUBLIC "-//IETF//DTD HTML 2.0//EN">
<html><head>
<title>502 Proxy Error</title>
</head><body>
<h1>Proxy Error</h1>
<p>The proxy server received an invalid
response from an upstream server.<br />
The proxy server could not handle the request<p>Reason: <strong>Error reading from remote server</strong></p></p>
</body></html>
Checkpoint saved. Resume from index 49


InternalServerError: <!DOCTYPE HTML PUBLIC "-//IETF//DTD HTML 2.0//EN">
<html><head>
<title>502 Proxy Error</title>
</head><body>
<h1>Proxy Error</h1>
<p>The proxy server received an invalid
response from an upstream server.<br />
The proxy server could not handle the request<p>Reason: <strong>Error reading from remote server</strong></p></p>
</body></html>

In [ ]:
##### using only RAG to answer questions

# answer_llm = OpenAIServerModel(
#     model_id=model_name,
#     api_base="https://generativelanguage.googleapis.com/v1beta/openai/",
#     api_key=os.getenv("Gemini_API_KEY"),
#     # flatten_messages_as_text=True,
#     temperature=0.2)

answer_llm = OpenAIServerModel(
    model_id=model_name,
    api_base="https://api.helmholtz-blablador.fz-juelich.de/v1",
    api_key=os.getenv("Blablador_API_KEY"),
    # flatten_messages_as_text=True,
    temperature=0.2)

print(f"The RAG uses the following model: {model_name}\n")

outputs_standard_rag = []

for example in tqdm(eval_dataset):
    question = example["question"]
    context = retriever_tool(question, k=7)

    prompt = f"""Given the question and supporting documents below, give a comprehensive answer to the question.
Respond only to the question asked, response should be concise and relevant to the question.
If the question is ambiguous or cannot be answered definitively, state so clearly.

Question:
{question}

{context}
"""
    messages = [{"role": "user", "content": prompt}]
    answer = answer_llm.generate(messages).content

    print("=======================================================")
    print(f"Question: {question}")
    # print(f"Context: {context}")
    print(f"Answer: {answer}")
    print(f'True answer: {example["answer"]}')

    results_agentic = {
        "question": question,
        "true_answer": example["answer"],
        "source_doc": example["source_doc"],
        "generated_answer": answer,
    }
    outputs_standard_rag.append(results_agentic)
    # time.sleep(5)

The RAG uses the following model: Qwen3 Coder 30B with function call



  2%|▏         | 1/65 [00:00<00:30,  2.07it/s]

Question: What architecture is the `tokenizers-linux-x64-musl` binary designed for?

Answer: The `tokenizers-linux-x64-musl` binary is designed for the **x86_64-unknown-linux-musl** architecture.
True answer: x86_64-unknown-linux-musl


  3%|▎         | 2/65 [00:01<00:38,  1.64it/s]

Question: What is the purpose of the BLIP-Diffusion model?

Answer: The purpose of the BLIP-Diffusion model is to enable **zero-shot subject-driven generation** and **control-guided zero-shot generation** in text-to-image synthesis. It leverages pre-trained subject representations to allow controllable and subject-specific image generation without requiring task-specific training. This is achieved through its architecture and training methodology detailed in the paper "BLIP-Diffusion: Pre-trained Subject Representation for Controllable Text-to-Image Generation and Editing".
True answer: The BLIP-Diffusion model is designed for controllable text-to-image generation and editing.


...

Question: What is the name of the open-source library created by Hugging Face to simplify Transformer acceleration?

Answer: The open-source library created by Hugging Face to simplify Transformer acceleration is called **Accelerate**.
True answer: Optimum


 98%|█████████▊| 64/65 [00:44<00:00,  1.91it/s]

Question: What parameter is used to ensure that elements in a row have the same height in Gradio?

Answer: The parameter used to ensure that elements in a row have the same height in Gradio is `equal_height`. This parameter should be passed to the `.style()` method of `gr.Row()`.
True answer: equal_height


100%|██████████| 65/65 [00:44<00:00,  1.45it/s]

Question: What is the command to install the latest version of Optimum with OpenVINO support?

Answer: The command to install the latest version of Optimum with OpenVINO support is:

```bash
pip install --upgrade-strategy eager optimum[openvino,nncf]
```
True answer: pip install --upgrade-strategy eager optimum["openvino"]


In [34]:
##### using vanilla LLM to answer questions

print(f"{model_name} are being used to answer questions.\n")

outputs_standard = []

for example in tqdm(eval_dataset):
    question = example["question"]

    prompt = f"""Answer the following question as clearly and concisely as possible.
Use your own knowledge to respond.
If the question is ambiguous or cannot be answered definitively, state so clearly.

Question:
{question}

"""
    messages = [{"role": "user", "content": prompt}]
    answer = answer_llm.generate(messages).content

    print("=======================================================")
    print(f"Question: {question}")
    print(f"Answer: {answer}")
    print(f'True answer: {example["answer"]}')

    results_llm = {
        "question": question,
        "true_answer": example["answer"],
        "source_doc": example["source_doc"],
        "generated_answer": answer,
    }
    outputs_standard.append(results_llm)
    time.sleep(5)

Qwen3 Coder 30B with function call are being used to answer questions.



  0%|          | 0/65 [00:00<?, ?it/s]

Question: What architecture is the `tokenizers-linux-x64-musl` binary designed for?

Answer: The `tokenizers-linux-x64-musl` binary is designed for:

- **Architecture**: x86_64 (64-bit Intel/AMD)
- **Operating System**: Linux
- **C runtime**: musl libc (rather than glibc)

This is a Linux binary compiled for 64-bit x86 architecture using the musl C library instead of the more common glibc. The "musl" designation indicates it's built against the musl libc library, which is commonly used in Alpine Linux and other minimal Linux distributions.
True answer: x86_64-unknown-linux-musl


  2%|▏         | 1/65 [00:05<06:14,  5.86s/it]

Question: What is the purpose of the BLIP-Diffusion model?

Answer: The BLIP-Diffusion model is designed for **image generation from text descriptions**. It combines the BLIP (Bootstrapped Language-Image Pretraining) vision-language model with diffusion models to create high-quality images based on textual prompts.

Its key purposes include:
- Generating realistic images from natural language descriptions
- Bridging the gap between text and image modalities
- Enabling controllable image synthesis through text guidance

The model leverages pre-trained vision-language understanding (from BLIP) and extends it with diffusion-based generation capabilities to produce diverse, high-fidelity images from text inputs.
True answer: The BLIP-Diffusion model is designed for controllable text-to-image generation and editing.


...

 97%|█████████▋| 63/65 [06:12<00:11,  6.00s/it]

Question: What parameter is used to ensure that elements in a row have the same height in Gradio?

Answer: In Gradio, the `elem_height` parameter is used to ensure that elements in a row have the same height.

However, it's worth noting that Gradio's layout system primarily relies on CSS flexbox properties, and the `elem_height` parameter specifically controls the height of individual components. For more complex row alignment, developers typically use `gr.Row()` containers with appropriate spacing and sizing parameters, or custom CSS styling.

The most direct answer is: **elem_height**
True answer: equal_height


 98%|█████████▊| 64/65 [06:18<00:05,  5.92s/it]

Question: What is the command to install the latest version of Optimum with OpenVINO support?

Answer: The command to install the latest version of Optimum with OpenVINO support is:

```bash
pip install optimum[openvino]
```

This installs the main Optimum library along with OpenVINO dependencies. For the latest development version, you can use:

```bash
pip install git+https://github.com/huggingface/optimum.git[openvino]
```

Note: Make sure you have Python 3.8+ and pip installed before running these commands.
True answer: pip install --upgrade-strategy eager optimum["openvino"]


100%|██████████| 65/65 [06:24<00:00,  5.91s/it]


In [35]:
EVALUATION_PROMPT = """You are a fair evaluator language model.

You will be given an instruction, a response to evaluate, a reference answer that gets a score of 3, and a score rubric representing a evaluation criteria are given.
1. Write a detailed feedback that assess the quality of the response strictly based on the given score rubric, not evaluating in general.
2. After writing a feedback, write a score that is an integer between 1 and 3. You should refer to the score rubric.
3. The output format should look as follows: \"Feedback: {{write a feedback for criteria}} [RESULT] {{an integer number between 1 and 3}}\"
4. Please do not generate any other opening, closing, and explanations. Be sure to include [RESULT] in your output.
5. Do not score conciseness: a correct answer that covers the question should receive max score, even if it contains additional useless information.

The instruction to evaluate:
{instruction}

Response to evaluate:
{response}

Reference Answer (Score 3):
{reference_answer}

Score Rubrics:
[Is the response complete, accurate, and factual based on the reference answer?]
Score 1: The response is completely incomplete, inaccurate, and/or not factual.
Score 2: The response is somewhat complete, accurate, and/or factual.
Score 3: The response is completely complete, accurate, and/or factual.

Feedback:"""

In [43]:
##### evaluation the performance of 3 different systems

evaluation_llm_name = "gemini-2.0-flash"
evaluation_llm = OpenAIServerModel(
    model_id=evaluation_llm_name,
    api_base="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=os.getenv("Gemini_API_KEY"),
    temperature=0)

results = {}
for system_type, outputs in [
    ("agentic_rag", outputs_agentic_rag),
    ("standard_rag", outputs_standard_rag),
    ("standard", outputs_standard),
]:
    print("evaluating {}".format(system_type))

    for experiment in tqdm(outputs):
        eval_prompt = EVALUATION_PROMPT.format(
            instruction=experiment["question"],
            response=experiment["generated_answer"],
            reference_answer=experiment["true_answer"],
        )
        messages = [
            {
                "role": "system",
                "content": "You are a fair evaluator language model."
            },
            {
                "role": "user",
                "content": eval_prompt
            },
        ]

        eval_result = evaluation_llm.generate(messages).content

        try:
            feedback, score = [
                item.strip() for item in eval_result.split("[RESULT]")
            ]
            experiment["eval_score_LLM_judge"] = score
            experiment["eval_feedback_LLM_judge"] = feedback
        except:
            print(f"Parsing failed - output was: {eval_result}")

        time.sleep(4)

    results[system_type] = pd.DataFrame.from_dict(outputs)
    results[system_type] = results[
        system_type].loc[~results[system_type]["generated_answer"].str.
                         contains("Error", na=False)]

evaluating agentic_rag


100%|██████████| 65/65 [05:13<00:00,  4.82s/it]


evaluating standard_rag


100%|██████████| 65/65 [06:06<00:00,  5.63s/it]


evaluating standard


 83%|████████▎ | 54/65 [04:24<00:53,  4.90s/it]


RateLimitError: Error code: 429 - [{'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 200, model: gemini-2.0-flash\nPlease retry in 26.234569858s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-2.0-flash', 'location': '

In [50]:
import time
import re
from tqdm import tqdm


def extract_retry_delay(error_message):
    """Extract retry delay from error message."""
    match = re.search(r'retry in ([\d.]+)s', str(error_message))
    if match:
        return float(match.group(1))
    return None


def evaluate_with_retry(evaluation_llm, messages, max_retries=5):
    """
    Evaluate with automatic retry on rate limit errors.
    
    Args:
        evaluation_llm: The LLM model instance
        messages: Messages to send to the model
        max_retries: Maximum number of retry attempts
    
    Returns:
        The evaluation result content
    """
    for attempt in range(max_retries):
        try:
            eval_result = evaluation_llm.generate(messages).content
            return eval_result
        except Exception as e:
            error_str = str(e)

            # Check if it's a rate limit error
            if "429" in error_str or "RESOURCE_EXHAUSTED" in error_str:
                # Extract suggested delay from error message
                retry_delay = extract_retry_delay(error_str)

                if retry_delay is None:
                    # Use exponential backoff if no delay specified
                    retry_delay = min(2**attempt * 5, 120)  # Max 2 minutes

                if attempt < max_retries - 1:
                    print(
                        f"\nRate limit hit. Waiting {retry_delay:.1f} seconds before retry {attempt + 1}/{max_retries}..."
                    )
                    time.sleep(retry_delay)
                else:
                    print(f"\nMax retries reached. Skipping this evaluation.")
                    raise
            else:
                # If it's not a rate limit error, raise immediately
                print(f"\nNon-rate-limit error occurred: {error_str}")
                raise

    return None


##### Evaluation of 3 different systems
evaluation_llm_name = "gemini-2.0-flash"
evaluation_llm = OpenAIServerModel(
    model_id=evaluation_llm_name,
    api_base="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=os.getenv("Gemini_API_KEY"),
    temperature=0)

results = {}
for system_type, outputs in [
    ("agentic_rag", outputs_agentic_rag),
    ("standard_rag", outputs_standard_rag),
    ("standard", outputs_standard),
]:
    print(f"\n{'='*50}")
    print(f"Evaluating {system_type}")
    print(f"{'='*50}")

    for experiment in tqdm(outputs):
        eval_prompt = EVALUATION_PROMPT.format(
            instruction=experiment["question"],
            response=experiment["generated_answer"],
            reference_answer=experiment["true_answer"],
        )
        messages = [
            {
                "role": "system",
                "content": "You are a fair evaluator language model."
            },
            {
                "role": "user",
                "content": eval_prompt
            },
        ]

        try:
            eval_result = evaluate_with_retry(evaluation_llm, messages)

            if eval_result:
                try:
                    feedback, score = [
                        item.strip() for item in eval_result.split("[RESULT]")
                    ]
                    experiment["eval_score_LLM_judge"] = score
                    experiment["eval_feedback_LLM_judge"] = feedback
                except:
                    print(f"\nParsing failed - output was: {eval_result}")
                    experiment["eval_score_LLM_judge"] = None
                    experiment["eval_feedback_LLM_judge"] = None

            # Small delay between requests to avoid hitting rate limits
            time.sleep(5)

        except Exception as e:
            print(f"\nFailed to evaluate experiment: {e}")
            experiment["eval_score_LLM_judge"] = None
            experiment["eval_feedback_LLM_judge"] = None

    results[system_type] = pd.DataFrame.from_dict(outputs)
    results[system_type] = results[
        system_type].loc[~results[system_type]["generated_answer"].str.
                         contains("Error", na=False)]

    print(
        f"\nCompleted {system_type}: {len(results[system_type])} valid results"
    )

print("\n" + "=" * 50)
print("Evaluation complete!")
print("=" * 50)


Evaluating agentic_rag


100%|██████████| 65/65 [06:19<00:00,  5.84s/it]



Completed agentic_rag: 65 valid results

Evaluating standard_rag


100%|██████████| 65/65 [06:23<00:00,  5.90s/it]



Completed standard_rag: 65 valid results

Evaluating standard


100%|██████████| 65/65 [06:31<00:00,  6.02s/it]


Completed standard: 64 valid results

Evaluation complete!


In [ ]:
print(results['standard'].question)

0     What architecture is the `tokenizers-linux-x64...
1     What is the purpose of the BLIP-Diffusion mode...
2     How can a user claim authorship of a paper on ...
3     What is the purpose of the /healthcheck endpoi...
4     What is the default context window size for Lo...
                            ...                        
60    What is the maximum size of a model checkpoint...
61    What is the purpose of Weights and Biases (W&B...
62    What is the name of the open-source library cr...
63    What parameter is used to ensure that elements...
64    What is the command to install the latest vers...
Name: question, Length: 64, dtype: object


In [ ]:
##### Show the accuracy of 3 different systems
DEFAULT_SCORE = 2  # Give average score whenever scoring fails


def fill_score(x):
    try:
        return int(x)
    except:
        return DEFAULT_SCORE


for system_type, outputs in [
    ("agentic_rag", outputs_agentic_rag),
    ("standard_rag", outputs_standard_rag),
    ("standard", outputs_standard),
]:

    results[system_type]["eval_score_LLM_judge_int"] = (
        results[system_type]["eval_score_LLM_judge"].fillna(
            DEFAULT_SCORE).apply(fill_score))
    results[system_type]["eval_score_LLM_judge_int"] = (
        results[system_type]["eval_score_LLM_judge_int"] - 1) / 2

    print(
        f"Average score for {system_type} : {results[system_type]['eval_score_LLM_judge_int'].mean()*100:.1f}%"
    )

In [ ]:
# save results

# Create a dictionary of JSON strings
json_results = {}
results_dir = "results"

for system_type, df in results.items():
    json_results[system_type] = json.loads(df.to_json(orient='records'))
save_json_path = os.path.join(results_dir, model_name + "_vect400_t0.2.json")

# Save to a file
with open(save_json_path, 'w') as f:
    json.dump(json_results, f, indent=4)

In [ ]:
# load results

results_dir = "results"
load_model_name = "gemini-2.5-flash-preview-05-20"
load_json_path = os.path.join(results_dir,
                              load_model_name + "_vect400_t0.2.json")

# Read the JSON file
with open(load_json_path, 'r') as f:
    loaded_json = json.load(f)

# Convert back to a dictionary of DataFrames
loaded_results = {}
for system_type, records in loaded_json.items():
    loaded_results[system_type] = pd.DataFrame.from_records(records)

results = loaded_results